# Portfolio follow-up checks

Extra checks run after the main analysis (`tabl_SaaS_Revenue-Churn_Analysis.ipynb`) for the portfolio write-up at
[portfolio-mo.vercel.app/projects/saas-revenue-churn-analysis](https://portfolio-mo.vercel.app/projects/saas-revenue-churn-analysis).

This notebook reads `subscriptions_enriched.csv`, which already holds the `tenure_months` and `risk_score` columns saved by the main notebook, so the numbers match it exactly.

In [1]:
import pandas as pd
from sklearn.metrics import roc_auc_score

subs = pd.read_csv('../data/subscriptions_enriched.csv')
mrr = pd.read_csv('../data/monthly_revenue.csv')
subs['is_churned'] = subs['churned'] == 'Yes'
active = subs[~subs['is_churned']]
churned = subs[subs['is_churned']]

## 1. Does annual billing lower churn within each plan?

Overall, monthly customers churn at 60.5% and annual at 40.3%. Cheaper plans skew monthly, so check the gap inside each plan.

In [2]:
plan_order = ['Starter', 'Professional', 'Business', 'Enterprise']
by_plan_billing = (
    subs.pivot_table(index='plan', columns='billing_cycle', values='is_churned', aggfunc='mean')
    .loc[plan_order] * 100
).round(1)
by_plan_billing['gap_pts'] = (by_plan_billing['Monthly'] - by_plan_billing['Annual']).round(1)
by_plan_billing

billing_cycle,Annual,Monthly,gap_pts
plan,,,
Starter,60.2,76.9,16.7
Professional,35.1,57.6,22.5
Business,27.4,52.9,25.5
Enterprise,22.2,21.9,-0.3


In [3]:
# Customer counts behind each rate
pd.crosstab(subs['plan'], subs['billing_cycle']).loc[plan_order]

billing_cycle,Annual,Monthly
plan,,
Starter,83,134
Professional,74,99
Business,73,87
Enterprise,18,32


The gap holds within Starter, Professional, and Business (17 to 26 points), so it isn't just plan mix. Enterprise is about 22% either way.

## 2. Churn by feature-usage band and by NPS score

In [4]:
bands = pd.cut(
    subs['feature_usage_pct'], bins=[0, 20, 40, 60, 80, 101], right=False,
    labels=['Under 20%', '20-39%', '40-59%', '60-79%', '80%+']
)
subs.groupby(bands, observed=True)['is_churned'].agg(customers='size', churned='sum', churn_rate='mean').assign(
    churn_rate=lambda d: (d['churn_rate'] * 100).round(1)
)

,customers,churned,churn_rate
feature_usage_pct,,,
Under 20%,107,94,87.9
20-39%,243,163,67.1
40-59%,127,56,44.1
60-79%,58,0,0.0
80%+,65,0,0.0


In [5]:
subs.groupby('nps_score')['is_churned'].agg(customers='size', churn_rate='mean').assign(
    churn_rate=lambda d: (d['churn_rate'] * 100).round(1)
)

,customers,churn_rate
nps_score,,
1,102,77.5
2,80,72.5
3,84,73.8
4,72,55.6
5,62,54.8
6,70,57.1
7,37,0.0
8,32,0.0
9,38,0.0


No customer using 60% or more of the features churned, and no customer with an NPS survey score of 7 or higher churned.

## 3. How much of the risk score's AUC comes from tenure?

Churned customers stop accruing tenure when they leave, while active customers keep accruing it, so tenure is partly a result of churn rather than a predictor of it. Compare the AUC with and without it.

In [6]:
y = subs['is_churned'].astype(int)
risk_feature = 1 - subs['feature_usage_pct'] / 100
risk_nps = 1 - subs['nps_score'] / 10

auc_saved = roc_auc_score(y, subs['risk_score'])
# Same 50:30 weighting of usage and NPS, rescaled to sum to 1
auc_no_tenure = roc_auc_score(y, risk_feature * 0.625 + risk_nps * 0.375)

print(f'AUC, composite score (usage, NPS, tenure): {auc_saved:.3f}')
print(f'AUC, usage and NPS only:                   {auc_no_tenure:.3f}')
for col in ['feature_usage_pct', 'nps_score', 'tenure_months']:
    print(f'AUC, {col} alone: {roc_auc_score(y, -subs[col]):.3f}')

AUC, composite score (usage, NPS, tenure): 0.937
AUC, usage and NPS only:                   0.914
AUC, feature_usage_pct alone: 0.840
AUC, nps_score alone: 0.786
AUC, tenure_months alone: 0.786


## 4. Monthly revenue held by at-risk customers

In [7]:
threshold = churned['feature_usage_pct'].mean() + churned['feature_usage_pct'].std()
low_usage = active[active['feature_usage_pct'] < threshold]
composite = active[active['risk_score'] > churned['risk_score'].mean()]

print(f'Usage threshold: {threshold:.2f}%')
print(f'Low-usage active customers: {len(low_usage)}, MRR ${low_usage["monthly_revenue"].sum():,.0f}')
print(f'Composite-score flags:      {len(composite)}, MRR ${composite["monthly_revenue"].sum():,.0f}')
print(f'All active customers:       {len(active)}, MRR ${active["monthly_revenue"].sum():,.0f}')

Usage threshold: 38.85%
Low-usage active customers: 85, MRR $81,858
Composite-score flags:      14, MRR $11,514
All active customers:       287, MRR $295,420


## 5. Did revenue growth slow in 2025?

In [8]:
mrr['month'] = pd.to_datetime(mrr['month'])
dec_rows = mrr[mrr['month'].dt.month == 12]
december = pd.Series(dec_rows['total_mrr'].values, index=dec_rows['month'].dt.year)
growth = pd.DataFrame({'dec_mrr': december, 'yoy_growth_pct': (december.pct_change() * 100).round(1)})
growth

,dec_mrr,yoy_growth_pct
month,,
2022,96100.66,NaN
2023,189579.35,97.3
2024,260856.24,37.6
2025,292628.61,12.2


In [9]:
mar_2025 = mrr.loc[mrr['month'] == '2025-03-01', 'total_mrr'].item()
dec_2025 = mrr.loc[mrr['month'] == '2025-12-01', 'total_mrr'].item()
print(f'Average monthly MRR gain in 2025: ${(december[2025] - december[2024]) / 12:,.0f} (vs ~$6,466/month on the full-period trend)')
print(f'MRR change, Mar to Dec 2025: {(dec_2025 / mar_2025 - 1) * 100:.1f}%')
mrr.loc[mrr['month'].between('2025-03-01', '2025-06-01'), ['month', 'churned_customers', 'monthly_churn_rate_pct']]

Average monthly MRR gain in 2025: $2,648 (vs ~$6,466/month on the full-period trend)
MRR change, Mar to Dec 2025: 4.2%


,month,churned_customers,monthly_churn_rate_pct
38,2025-03-01,7,2.72
39,2025-04-01,17,6.30
40,2025-05-01,15,5.68
41,2025-06-01,11,4.21


MRR grew 38% in 2024 but 12% in 2025, and only about 4% from March to December 2025. April and May 2025 lost 32 customers, the largest two-month churn in the data. The three-year linear trend (R² 0.97) describes the past well but overstates where revenue is heading.